# オプション課題：何枚あれば学習できるか（FashionMNIST）

学習に使う画像の枚数を 100〜48,000 枚に変えると，正解率はどう変わるか．MLP と CNN で違いはあるか．
**やってみるまで分からない．** 3〜4 グループで組み，12 の条件（枚数 6 通り × MLP／CNN）を分担して，全員分で 1 枚のグラフを作る．

1. 実行する前に，下の (1) に予測を書く
2. (2) に自分の担当（MODEL と N_TRAIN）を書き，(3) を実行する
3. (4) の手順で CSV を仲間と共有する
4. (5) でグラフを描き，(6) の考察を書く（提出）


## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- 100 枚のとき，テスト正解率は何%くらいか：MLP ＿＿％，CNN ＿＿％
- 48,000 枚（全部）のとき：MLP ＿＿％，CNN ＿＿％
- 枚数を増やすと正解率はどう増えるか（まっすぐ／だんだん鈍る／その他）：＿＿


In [ ]:
# (2) 自分の担当を書く（分担表で決めたもの）
MODEL = "MLP"      # "MLP" か "CNN"
N_TRAIN = 1000     # 100, 300, 1000, 3000, 10000, 48000 のどれか

In [ ]:
# (3) 学習して，テストデータ（1 万枚）の正解率を記録する．乱数の種はグループ番号
from common.device import get_device
from common.datasize import run
from common.logger import ResultLogger

device = get_device()
r = run(MODEL, N_TRAIN, seed=GROUP_ID, device=device)
print(f"{MODEL}  {N_TRAIN:,} 枚  テスト正解率 {r['test_accuracy']*100:.2f}%  学習時間 {r['train_sec']:.1f} 秒")

logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="opt1", device=device)
logger.log_many({"test_accuracy": r["test_accuracy"], "train_sec": r["train_sec"]},
                condition=f"{MODEL}_{N_TRAIN}", seed=GROUP_ID)
print("書き込み先:", logger.path)

## (4) 共有する

1. 自分の `results/c1_d1_opt1_<グループ>.csv` を，組んだ仲間全員に送る（Slack・AirDrop など）
2. 仲間から受け取った CSV は，すべて `results/opt_shared/` フォルダに入れる（フォルダが無ければ作る．同じ名前のファイルは末尾に `_2` などを付けて別名にする）
3. 担当を変えてもう 1 条件実行したいときは，(2) を書き換えて (3) を実行し直す（同じ CSV に行が増える）


In [ ]:
# (5) 集まった全員分をグラフにする（何回実行してもよい）
import glob
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # Mac の日本語フォント（文字化け防止）

files = glob.glob(str(ROOT / "results" / "c1_d1_opt1_*.csv")) + glob.glob(str(ROOT / "results" / "opt_shared" / "*.csv"))
df = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
acc = df[df["metric_name"] == "test_accuracy"].copy()
acc["model"] = acc["condition"].str.split("_").str[0]
acc["n_train"] = acc["condition"].str.split("_").str[1].astype(int)
acc["acc"] = acc["metric_value"] * 100

plt.figure(figsize=(8, 5))
for m, color in [("MLP", "tab:blue"), ("CNN", "tab:orange")]:
    g = acc[acc["model"] == m]
    plt.scatter(g["n_train"], g["acc"], color=color, alpha=0.6, label=f"{m}（各回）")
    mean = g.groupby("n_train")["acc"].mean()
    plt.plot(mean.index, mean.values, color=color, marker="o", label=f"{m}（平均）")
plt.xscale("log")
plt.xlabel("学習に使った枚数（対数目盛）")
plt.ylabel("テスト正解率（%）")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

done = acc.groupby(["model", "n_train"]).size()
print(f"集まった条件：{len(done)} / 12　（ファイル {len(files)} 個，実行 {len(acc)} 回）")
print(done.unstack(0).fillna(0).astype(int))

## (6) 考察（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．

1. 予測と比べて，当たったこと・外れたこと：＿＿
2. 枚数を 10 倍にすると，正解率は何ポイント上がったか．どの枚数から伸びが鈍るか：＿＿
3. MLP と CNN の差は，枚数によってどう変わったか．同じ条件を複数のグループが実行した場合，値はどれくらいぶれたか：＿＿
4. GW3 のオリジナル画像（訓練 140 枚）の正解率が FashionMNIST より低かった理由を，このグラフから説明する：＿＿
